# Process All 255 — Debug V5

In [ ]:
# Cell 1: Setup
import os, sys, json, subprocess, warnings, shutil
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('Installing deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yt-dlp', '-q'], capture_output=True)
print('✓ yt-dlp')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from transformers import AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
print('✓ All imports')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

WORK = '/kaggle/working/process255'
INPUT = '/kaggle/input'
os.makedirs(f'{WORK}/features', exist_ok=True)
os.makedirs(f'{WORK}/audio', exist_ok=True)
os.makedirs(f'{WORK}/labels', exist_ok=True)

# Check input datasets
for d in os.listdir(INPUT):
    subpath = os.path.join(INPUT, d)
    if os.path.isdir(subpath):
        contents = os.listdir(subpath)[:5]
        print(f'  Input/{d}: {contents}')

# Find label directory
LABEL_DIR = None
for d in os.listdir(INPUT):
    p = os.path.join(INPUT, d, 'labels')
    if os.path.isdir(p) and len(os.listdir(p)) > 0:
        LABEL_DIR = p
        break
if LABEL_DIR:
    print(f'✓ Labels found: {LABEL_DIR} ({len(os.listdir(LABEL_DIR))} files)')
else:
    print('⚠️ No labels directory found in inputs!')

print('\nSetup complete')

In [ ]:
# Cell 2: Load 255 video IDs
ALL_VIDEOS = [
    '-UPIA46hBZs','-vcKXr6WBNc','0AvUvJ_S2Os','0Pl51hxcK-o','0g7nezWZyfY',
    '0zpUnJSG0EQ','18H1aeoGybw','18rLwnvxOU0','1ILQmgHvtd4','1Uo27tH3JQ4',
    '1pPnJut3KLw','1tO9MWWOgHk','1u-pq9LLWlU','21gOjz-Xk7s','2SUfHIbT0HI',
    '2axWotdMFsw','2ql8QJWmNM8','3TgRGK1vrzs','3ZTClwMxpmM','3new05S61w4',
    '41piF6uPhXg','482LeT9UT7I','4ZiXvhSxnD4','53JXuJGmhoU','5bKcTy3zag4'
]  # First 25 for testing
print(f'Total videos this run: {len(ALL_VIDEOS)}')

In [ ]:
# Cell 3: Load models + define extractors
print('Loading WavLM...')
SR_WAVLM, SR_PROSODY = 16000, 22050
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device); wavlm.eval()
print('✓ WavLM loaded')

def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]; v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
                  np.std(f0c) if len(f0c)>0 else 0,
                  np.max(f0c) if len(f0c)>0 else 0,
                  np.min(f0c) if len(f0c)>0 else 0,
                  np.mean(v) if len(v)>0 else 0])
    except: f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1)])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except: f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_features(y16, y22, t0, t1):
    dur = t1 - t0
    if dur < 0.005: return None
    s16, e16 = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    c16 = y16[s16:e16]
    if len(c16) < int(0.01*SR_WAVLM): return None
    if len(c16) < 5*SR_WAVLM:
        c16 = np.pad(c16, (0, int(5*SR_WAVLM)-len(c16)))
    with torch.no_grad():
        wemb = wavlm(torch.tensor(c16/32768.0).unsqueeze(0).to(device)).last_hidden_state.mean(1).squeeze().cpu().numpy()
    s22, e22 = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    pros = prosody23(y22[s22:e22], SR_PROSODY)
    return np.concatenate([wemb, pros])

print('✓ Feature extractor defined')

In [ ]:
# Cell 4: Download Audio
def dl_audio(vid):
    for ext in ['.wav', '.m4a']:
        p = f'{WORK}/audio/{vid}{ext}'
        if os.path.exists(p): return p
    base = f'{WORK}/audio/{vid}'
    cmd = ['yt-dlp', '-f', 'bestaudio',
           '-o', f'{base}.%(ext)s',
           f'https://www.youtube.com/watch?v={vid}',
           '--no-playlist', '--quiet', '--socket-timeout', '60',
           '--extract-audio', '--audio-format', 'wav']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        if os.path.exists(f'{base}.wav'): return f'{base}.wav'
        if os.path.exists(f'{base}.m4a'): return f'{base}.m4a'
    except: pass
    return None

print(f'Downloading {len(ALL_VIDEOS)} audio files...')
ok, fail = 0, 0
for vid in tqdm(ALL_VIDEOS):
    result = dl_audio(vid)
    if result: ok += 1
    else: fail += 1

print(f'Downloaded: {ok}, Failed: {fail}')

In [ ]:
# Cell 5: Extract Features (DEBUG)
DONE_FILE = f'{WORK}/features_done.json'
done = set()
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f: done = set(json.load(f))
print(f'Resuming: {len(done)} done')

processed, errors, no_labels = 0, [], []

for vid_idx, vid in enumerate(tqdm(ALL_VIDEOS)):
    if vid in done: continue
    
    # Find audio
    ap = None
    for ext in ['.wav', '.m4a']:
        p = f'{WORK}/audio/{vid}{ext}'
        if os.path.exists(p): ap = p; break
    if not ap: continue
    
    # Find label
    lp = f'{LABEL_DIR}/{vid}.csv' if LABEL_DIR else None
    if not lp or not os.path.exists(lp):
        no_labels.append(vid)
        continue
    
    try:
        y22, _ = librosa.load(ap, sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(ap, sr=SR_WAVLM, mono=True)
        df = pd.read_csv(lp)
        
        feats, lbls = [], []
        for _, row in df.iterrows():
            try:
                ts = eval(str(row['timestamp']))
                t0, t1 = float(ts[0]), float(ts[1])
                feat = word_features(y16, y22, t0, t1)
                lbl = str(row.get('label', 'O')).strip()
                if feat is not None:
                    feats.append(feat)
                    lbls.append(1 if lbl in ('B','I','L') else 0)
            except Exception as word_err:
                pass  # Skip bad words
        
        if feats:
            np.save(f'{WORK}/features/{vid}_features.npy', np.array(feats, dtype=np.float32))
            np.save(f'{WORK}/features/{vid}_labels.npy', np.array(lbls, dtype=np.int32))
            done.add(vid)
            processed += 1
            pos_rate = sum(lbls)/max(len(lbls),1)
            print(f'  ✓ {vid}: {len(feats)} words, {100*pos_rate:.1f}% laugh')
        
        # Save checkpoint every 5 videos
        if processed % 5 == 0:
            with open(DONE_FILE, 'w') as f:
                json.dump(sorted(done), f)
    except Exception as e:
        errors.append((vid, str(e)))
        print(f'  ❌ {vid}: {e}')

with open(DONE_FILE, 'w') as f:
    json.dump(sorted(done), f)

print(f'\n=== SUMMARY ===')
print(f'Processed: {processed}/{len(ALL_VIDEOS)}')
print(f'No labels: {len(no_labels)} -> {no_labels[:5]}')
print(f'Errors: {len(errors)}')
for v, e in errors[:5]:
    print(f'  {v}: {e}')